# AIMO3 - TIR + RAG (Meta-Harness Inspired)

**Model**: GPT-OSS-120B via vLLM on H100
**Retrieval**: BM25 over 6.4K math problems → top-3 few-shot examples
**Inference**: TIR (code execution) + weighted majority voting
**API**: kaggle_evaluation.aimo_3_inference_server

In [ ]:
# ============================================================
# Cell 0: Install vLLM + deps from bundled wheels
# Exact pattern from andreasbis/aimo-3-gpt-oss-120b-with-tools
# ============================================================
import os, sys, subprocess

temp_dir = '/kaggle/tmp/setup'
archive = '/kaggle/input/aimo-3-utils/wheels.tar.gz'

if not os.path.exists(temp_dir):
    os.makedirs(temp_dir, exist_ok=True)
    subprocess.run(['tar', '-xzf', archive, '-C', temp_dir], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--no-index', '--find-links', f'{temp_dir}/wheels',
    'vllm'
], check=True)

print('vLLM installed.')

In [ ]:
# ============================================================
# Cell 1: Imports & Configuration
# ============================================================
import os, sys, time, re, json, traceback, threading, gc, math
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr
from collections import Counter, defaultdict
from pathlib import Path

import polars as pl

# === CONFIGURATION ===
MODEL_PATH = '/kaggle/input/gpt-oss-120b/transformers/default/1'
CORPUS_PATH = '/kaggle/input/aimo3-math-retrieval-corpus/math_corpus.jsonl'
N_SAMPLES = 32
MAX_TOKENS = 8192
TEMPERATURE = 0.7
TOP_P = 0.95
CODE_TIMEOUT = 30
TIR_MAX_RETRIES = 3
ANSWER_MOD = 100_000
RAG_TOP_K = 3

print(f'Model: {MODEL_PATH}')
print(f'Corpus: {CORPUS_PATH}')
print(f'N={N_SAMPLES}, tokens={MAX_TOKENS}, temp={TEMPERATURE}, RAG_K={RAG_TOP_K}')

In [ ]:
# ============================================================
# Cell 2: BM25 Retriever (loads immediately, no GPU needed)
# Meta-Harness pattern: math-aware tokenizer + domain routing
# ============================================================

# Math-aware tokenizer from Meta-Harness paper
_MATH_TOKEN = re.compile(r'\\[a-zA-Z]+|[_^]\{[^}]*\}|[a-zA-Z]+|[0-9]+|\S')

def math_tokenize(text):
    t = text.lower().replace('\\leqslant','\\le').replace('\\geqslant','\\ge')
    t = t.replace('\\tfrac','\\frac').replace('\\dfrac','\\frac')
    t = t.replace('\\left','').replace('\\right','').replace('\\displaystyle','')
    return _MATH_TOKEN.findall(t)

# Domain classifier
GEO = ['triangle','circle','angle','perpendicular','inscribed','tangent','chord',
       'incircle','circumcircle','orthocenter','altitude','midpoint','polygon']
NT = ['prime','divisible','modulo','gcd','remainder','congruent','coprime',
      'fermat','euler','residue','diophantine']
COMBO = ['how many','number of ways','permutation','combinat','probability',
         'expected value','pigeonhole','coloring','partition','subset']

def classify_domain(problem):
    p = problem.lower()
    scores = {'geometry': sum(1 for k in GEO if k in p),
              'number_theory': sum(1 for k in NT if k in p),
              'combinatorics': sum(1 for k in COMBO if k in p)}
    for d in ['combinatorics','geometry','number_theory']:
        if scores[d] >= 2: return d
    best = max(scores, key=scores.get)
    return best if scores[best] >= 1 else 'algebra'

# Solution truncation limits per domain (from Meta-Harness)
SOL_MAX = {'combinatorics':800, 'geometry':300, 'number_theory':400, 'algebra':400}

# Load corpus
corpus = []
if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH) as f:
        for line in f:
            if line.strip():
                corpus.append(json.loads(line))
    print(f'Loaded {len(corpus)} problems')
else:
    print(f'WARNING: Corpus not found at {CORPUS_PATH}')

# Build BM25 index (simple token overlap - no external deps needed)
corpus_tokens = [set(math_tokenize(p['problem'])) for p in corpus]

def retrieve(problem, k=RAG_TOP_K, target_diff=7.0):
    """Retrieve top-k similar problems with diversity filtering."""
    if not corpus:
        return []
    query_tokens = set(math_tokenize(problem))
    if not query_tokens:
        return []
    
    # Score by token overlap (BM25-lite)
    scored = []
    for i, ct in enumerate(corpus_tokens):
        overlap = len(query_tokens & ct)
        if overlap > 0:
            # IDF-like: rarer shared tokens score higher
            score = overlap / math.sqrt(max(len(ct), 1))
            scored.append((i, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    
    # Take top 20, filter by difficulty, deduplicate
    candidates = scored[:20]
    
    # Prefer code solutions
    candidates.sort(key=lambda x: (corpus[x[0]].get('has_code', False), x[1]), reverse=True)
    
    # Jaccard diversity
    selected = []
    sel_tokens = []
    for idx, score in candidates:
        if len(selected) >= k:
            break
        ct = corpus_tokens[idx]
        too_similar = False
        for st in sel_tokens:
            jacc = len(ct & st) / max(len(ct | st), 1)
            if jacc > 0.5:
                too_similar = True
                break
        if not too_similar:
            selected.append(corpus[idx])
            sel_tokens.append(ct)
    return selected

def format_examples(examples, domain='algebra'):
    """Format retrieved examples as few-shot text."""
    if not examples:
        return ''
    max_c = SOL_MAX.get(domain, 400)
    parts = ['Here are some similar solved examples for reference:\n']
    for i, ex in enumerate(examples, 1):
        sol = ex['solution'][:max_c]
        if len(ex['solution']) > max_c:
            sol += '...'
        part = f'Example {i}:\nProblem: {ex["problem"]}\nSolution: {sol}'
        if ex.get('answer'):
            part += f'\nAnswer: {ex["answer"]}'
        parts.append(part)
    return '\n\n'.join(parts)

print(f'Retriever ready ({len(corpus)} problems indexed).')

In [ ]:
# ============================================================
# Cell 3: Answer Extraction
# ============================================================

def _parse_number(s):
    s = s.strip().replace('\\,','').replace('\\;','').replace('\\!','').replace(',','')
    s = re.sub(r'\\text\{.*?\}', '', s)
    s = re.sub(r'\\mathrm\{.*?\}', '', s)
    try: return int(s)
    except ValueError: pass
    try:
        f = float(s)
        if f == int(f) and f == f: return int(f)
    except (ValueError, OverflowError): pass
    m = re.search(r'(-?\d+(?:\.\d+)?)', s)
    if m:
        try:
            f = float(m.group(1))
            if f == int(f): return int(f)
        except: pass
    return None

def extract_boxed(text):
    idx = text.rfind('\\boxed')
    if idx == -1: return None
    bs = text.find('{', idx)
    if bs == -1: return None
    depth, end = 0, bs
    for i in range(bs, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0: end = i; break
    return _parse_number(text[bs+1:end].strip())

def extract_code_output(text):
    ms = re.findall(r'```output\s*(.*?)```', text, re.DOTALL)
    return _parse_number(ms[-1].strip()) if ms else None

def extract_nl(text):
    for p in [r'(?:the\s+)?(?:final\s+)?answer\s+is\s*[:\s]*(-?\d+(?:\.\d+)?)',
              r'(?:therefore|thus|hence|so)\s*,?\s*(?:the\s+answer\s+is\s+)?(-?\d+(?:\.\d+)?)',
              r'answer\s*[=:]\s*(-?\d+(?:\.\d+)?)']:
        ms = re.findall(p, text, re.IGNORECASE)
        if ms: return _parse_number(ms[-1])
    return None

def extract_last_int(text):
    ms = re.findall(r'(?<![.\d])(-?\d+)(?![.\d])', text)
    return _parse_number(ms[-1]) if ms else None

def extract_answer(text):
    for fn in [extract_boxed, extract_code_output, extract_nl, extract_last_int]:
        r = fn(text)
        if r is not None: return r % ANSWER_MOD
    return None

print('Answer extraction ready.')

In [ ]:
# ============================================================
# Cell 4: Code Execution Sandbox
# ============================================================

SANDBOX_IMPORTS = '''
import math
import numpy as np
import sympy as sp
from sympy import *
from sympy import symbols, solve, simplify, expand, factor, Rational, sqrt, oo
from sympy import pi, E, I, sin, cos, tan, log, exp, Abs, floor, ceiling
from sympy import gcd, lcm, isprime, nextprime, factorint, divisors, totient
from sympy import binomial, factorial, fibonacci, Matrix, det, eye
from sympy.ntheory import mobius, primerange
from itertools import combinations, permutations, product as iproduct
from collections import Counter, defaultdict
from fractions import Fraction
from functools import reduce
import itertools
'''

def execute_code(code, timeout=CODE_TIMEOUT):
    stdout_buf, stderr_buf = StringIO(), StringIO()
    ns = {}
    try: exec(SANDBOX_IMPORTS, ns)
    except: pass
    exc_holder = [None]
    def _run():
        try:
            with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
                exec(code, ns)
        except Exception as e: exc_holder[0] = e
    t = threading.Thread(target=_run, daemon=True)
    t.start(); t.join(timeout=timeout)
    if t.is_alive(): return '', f'Timeout after {timeout}s', False
    so, se = stdout_buf.getvalue(), stderr_buf.getvalue()
    if exc_holder[0]:
        e = exc_holder[0]
        return so, ''.join(traceback.format_exception(type(e), e, e.__traceback__)), False
    return so, se, True

def extract_code_blocks(text):
    blocks = re.findall(r'```python\s*\n(.*?)```', text, re.DOTALL)
    return blocks if blocks else re.findall(r'```\s*\n(.*?)```', text, re.DOTALL)

print('Code sandbox ready.')

In [ ]:
# ============================================================
# Cell 5: Lazy Model Loader + Prompt Builder
# Model loads on first predict() call (15-min startup deadline)
# ============================================================
from vllm import LLM, SamplingParams

class LazyModel:
    def __init__(self):
        self._llm = None
        self._tokenizer = None
    def _load(self):
        if self._llm: return
        print(f'Loading model from {MODEL_PATH}...')
        t0 = time.time()
        self._llm = LLM(
            model=MODEL_PATH, tensor_parallel_size=1,
            gpu_memory_utilization=0.92, max_model_len=16384,
            dtype='auto', kv_cache_dtype='fp8_e4m3',
            enable_prefix_caching=True, max_num_seqs=64,
            trust_remote_code=True,
        )
        self._tokenizer = self._llm.get_tokenizer()
        print(f'Model loaded in {time.time()-t0:.1f}s')
    @property
    def llm(self): self._load(); return self._llm
    @property
    def tokenizer(self): self._load(); return self._tokenizer

model = LazyModel()

TIR_SYSTEM = '''You are a world-class mathematician. Solve the given problem step by step.
- Write Python code in ```python ... ``` blocks. You will see output in ```output ... ``` blocks.
- Use sympy for symbolic computation, numpy for numerical work.
- After reaching the answer, put it inside \\boxed{N} where N is an integer.
- The answer must be a non-negative integer between 0 and 99999.
- Double-check your answer with a verification code block.'''

TYPE_HINT = {
    'algebra': 'This is an algebra problem. Use sympy.symbols() and sympy.solve().',
    'combinatorics': 'This is a combinatorics problem. Enumerate small cases first.',
    'geometry': 'This is a geometry problem. Set up coordinates. Use sympy.',
    'number_theory': 'This is a number theory problem. Use sympy.factorint(), pow(b,e,m).',
    'default': 'Solve step by step with Python code.',
}

def build_prompt(problem, domain='default', rag_text=''):
    hint = TYPE_HINT.get(domain, TYPE_HINT['default'])
    user_parts = [hint]
    if rag_text:
        user_parts.append(rag_text)
    user_parts.append(f'Problem:\n{problem}')
    msgs = [
        {'role': 'system', 'content': TIR_SYSTEM},
        {'role': 'user', 'content': '\n\n'.join(user_parts)},
    ]
    return model.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print('Model wrapper + prompt builder ready.')

In [ ]:
# ============================================================
# Cell 6: TIR Batch Solver
# ============================================================

def tir_solve_batch(prompt, n_samples=N_SAMPLES):
    params = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P,
                            max_tokens=MAX_TOKENS, n=n_samples, stop=['```output'])
    outputs = model.llm.generate([prompt], params, use_tqdm=False)
    completions = [o.text for o in outputs[0].outputs]
    results = []
    for comp in completions:
        full_text = comp
        code_executed = code_succeeded = False
        blocks = extract_code_blocks(comp)
        if blocks:
            so, se, ok = execute_code(blocks[-1])
            code_executed = True; code_succeeded = ok
            out = so.strip() if ok and so.strip() else (se.strip()[:500] if not ok else '(no output)')
            full_text += f'\n```output\n{out}\n```\n'
            cont_prompt = prompt + full_text
            for _ in range(TIR_MAX_RETRIES):
                cp = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS//2, n=1, stop=['```output'])
                co = model.llm.generate([cont_prompt], cp, use_tqdm=False)
                chunk = co[0].outputs[0].text if co[0].outputs else ''
                if not chunk.strip(): break
                full_text += chunk
                nb = extract_code_blocks(chunk)
                if nb:
                    so, se, ok = execute_code(nb[-1])
                    if ok: code_succeeded = True
                    out = so.strip() if ok and so.strip() else se.strip()[:500]
                    full_text += f'\n```output\n{out}\n```\n'
                    cont_prompt = prompt + full_text
                else: break
                if extract_boxed(full_text) is not None: break
        answer = extract_answer(full_text)
        results.append((answer, code_executed, code_succeeded, '\\boxed' in full_text))
    return results

print('TIR solver ready.')

In [ ]:
# ============================================================
# Cell 7: Voting + Time Manager
# ============================================================

def vote(results):
    answers, weights = [], []
    for ans, ce, co, bx in results:
        if ans is None: continue
        w = 1.0
        if ce and co: w += 2.0
        elif ce: w += 0.5
        if bx: w += 0.5
        answers.append(ans); weights.append(w)
    if not answers: return 0, 0.0
    wt = {}
    for a, w in zip(answers, weights): wt[a] = wt.get(a, 0.0) + w
    best = max(wt, key=wt.get)
    conf = Counter(answers).most_common(1)[0][1] / len(answers)
    return best % ANSWER_MOD, conf

class Timer:
    def __init__(self, total=32400, per=1700, n=110):
        self.total, self.per, self.n = total, per, n
        self.start = time.time(); self.solved = 0; self.times = []
    def elapsed(self): return time.time() - self.start
    def remaining(self): return max(0, self.total - self.elapsed())
    def budget(self):
        r = max(1, self.n - self.solved)
        return max(60.0, min(self.remaining() / r, self.per))
    def get_n(self, base=N_SAMPLES):
        b = self.budget()
        if b >= 1500: return base
        elif b >= 900: return max(16, base//2)
        elif b >= 300: return max(8, base//4)
        return 4
    def record(self, t): self.solved += 1; self.times.append(t)
    def skip(self): return self.remaining() < 30
    def status(self):
        a = sum(self.times)/len(self.times) if self.times else 0
        return f'{self.solved}/{self.n} | {self.elapsed():.0f}s | {self.remaining():.0f}s left | avg {a:.1f}s'

timer = Timer()
print('Voting + timer ready.')

In [ ]:
# ============================================================
# Cell 8: Main Solve (RAG + TIR + Vote)
# ============================================================

def solve(problem):
    if timer.skip(): return 0
    t0 = time.time()
    try:
        # 1. Classify domain
        domain = classify_domain(problem)
        
        # 2. RAG: retrieve similar solved problems
        examples = retrieve(problem, k=RAG_TOP_K)
        rag_text = format_examples(examples, domain)
        
        # 3. Build prompt with few-shot examples
        prompt = build_prompt(problem, domain, rag_text)
        
        # 4. Adaptive N
        n = timer.get_n()
        
        # 5. TIR solve
        results = tir_solve_batch(prompt, n_samples=n)
        
        # 6. Vote
        answer, conf = vote(results)
        
        valid = sum(1 for r in results if r[0] is not None)
        code_ok = sum(1 for r in results if r[2])
        elapsed = time.time() - t0
        timer.record(elapsed)
        
        print(f'  domain={domain} | rag={len(examples)} | n={n} | valid={valid} | '
              f'code_ok={code_ok} | conf={conf:.2f} | ans={answer} | {elapsed:.1f}s | {timer.status()}')
        return answer
    except Exception as e:
        elapsed = time.time() - t0
        timer.record(elapsed)
        print(f'  ERROR: {e} | {elapsed:.1f}s')
        traceback.print_exc()
        return 0

print('Solver ready.')

In [ ]:
# ============================================================
# Cell 9: Kaggle Submission Server
# MUST call serve() within 15 minutes of script start
# ============================================================
import kaggle_evaluation.aimo_3_inference_server
import glob

def predict(id_, problem):
    pid = id_.item(0)
    ptxt = problem.item(0)
    print(f'\n{"="*60}\nProblem {timer.solved+1} (id={pid}):')
    print(f'  {ptxt[:120]}...' if len(ptxt) > 120 else f'  {ptxt}')
    gc.disable()
    answer = solve(ptxt)
    gc.enable(); gc.collect()
    answer = int(answer) % ANSWER_MOD
    print(f'  => {answer}')
    return pl.DataFrame({'id': [pid], 'answer': [answer]})

server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    server.serve()
else:
    # Find test.csv - search all possible locations
    candidates = (
        glob.glob('/kaggle/input/competitions/*/test.csv') +
        glob.glob('/kaggle/input/*/test.csv') +
        glob.glob('/kaggle/input/*/*/test.csv')
    )
    if candidates:
        test_path = candidates[0]
    else:
        # Last resort: list what we have
        print("Could not find test.csv. Available paths:")
        for root, dirs, files in os.walk('/kaggle/input'):
            depth = root.replace('/kaggle/input', '').count(os.sep)
            if depth < 3:
                for f in files:
                    if f.endswith('.csv'):
                        print(f"  {os.path.join(root, f)}")
        test_path = '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv'
    
    print(f'Using test file: {test_path}')
    server.run_local_gateway((test_path,))

print(f'\nDone! {timer.status()}')